# 02 — Four signals on the US top 500, net of costs

**Hackathon 1 MaQI 2026-27 · team Ricardo (Aymen, Gauthier)**

## 1. Introduction

**The question.** Do four classic signal families (mean-reversion, momentum, quality, value) make
money **after transaction costs** on the 500 largest US stocks between 2011 and 2024? And can at
least one of them be improved?

**The setting.** Every team shares the same simulation rules, so that results are comparable.
Each signal is a cross-sectional rank. The portfolio is a quintile long-short, targeted at 10%
annualised volatility on $100m. A signal computed at close t is executed at close t+1. Costs are
linear (2, 5 and 10 bp of the notional traded), and every performance figure is net of those
costs.

**Conclusions in 3 lines** (details in §12):

1. Only the **slow** signals survive costs: momentum (net Sharpe 0.35 at 5 bp) and quality
   (0.19). Value loses money over this period, even before costs.
2. Mean-reversion has a real gross edge, but it **trades 300 times a year** and breaks even at
   1.2 bp. The 1/N blend inherits this and turns negative at 5 bp.
3. Our improvement (rebalancing mean-reversion every 4 days) halves turnover and makes the net
   Sharpe positive at 2 bp, but **not at 5 bp**. It was chosen in-sample, so it is fragile.

The point-in-time panel comes from `01_data.ipynb`, which must be run first. This notebook
then goes through data exploration (§3), data loopholes (§4) and feature design (§5) before the
signals (§6), the simulator (§7), the results (§8-§10), monitoring (§11) and the conclusion (§12).

## 2. Parameters and loading

Each parameter is commented with its justification. The simulator rules (quintiles, 10%
volatility, 63 days, costs of 2/5/10 bp) are shared by every team. The other parameters are our
choices.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Data -----------------------------------------------------------------------------------
CLEAN_PATH = Path("data/panel_clean.parquet")  # produced by 01_data.ipynb
TOTAL_RETURN = "ret_1d_total"  # the simulator earns the total return: dividends ≈ 2.5%/yr on this universe
FALLBACK = "ret_1d"            # price return only, if the total return were missing

# --- Data quality and features -------------------------------------------------------------
FUNDAMENTAL_FIELDS = ("CSHOQ", "CEQQ", "SEQQ", "ATQ", "DLTTQ", "DLCQ", "NIQ",
                      "OIADPQ", "REVTQ", "TXTQ", "ACTQ", "LCTQ")  # the 12 quarterly fields delivered
EXTREME_RETURN = 0.5       # |daily return| > 50% on a large cap: badly adjusted split, spin-off or genuine shock
FROZEN_DAYS = 5            # 5 identical closes in a row on a top-500 member: price probably frozen
MAX_GAP_DAYS = 7           # no return across a trading gap longer than a week (same rule as ret_1d)
QUARTERS = 4               # sum over 4 rolling quarters: a single quarter carries its seasonality
ADJACENT_DAYS = (60, 120)  # two consecutive quarter ends are ~91 days apart; outside this range a quarter is missing

# --- Signals --------------------------------------------------------------------------------
SIGNALS = ["mean_reversion", "momentum", "quality", "value"]
REVERSAL_LOOKBACK = 5    # one trading week: the overreaction horizon of the baseline formula
MOMENTUM_LOOKBACK = 252  # one trading year: the standard window (Jegadeesh-Titman, Asness et al. 2013)
MOMENTUM_SKIP = 21       # skip the last month, where short-term reversal dominates

# --- Simulator: shared rules ----------------------------------------------------------------
YEAR = 252               # trading days per year, to annualise
QUANTILES = 5            # long the top quintile, short the bottom one (rule 1)
SIGNAL_LAG = 1           # signal at close t → execution at close t+1 (rule 4)
TARGET_VOL = 0.10        # 10% annualised volatility (rule 3); on $100m and with linear costs, the AUM
                         # level does not change returns, so it does not appear
VOL_WINDOW = 63          # σ estimated over one quarter of P&L
MAX_LEVERAGE = 4.0       # a quiet quarter cannot lever the book without limit
COST_BPS = (2.0, 5.0, 10.0)  # 2 bp: liquid large cap; 10 bp: pessimistic desk (rule 5)
DEFAULT_BPS = 5.0        # the reference cost level in the charts

# --- Simulator: our choices -----------------------------------------------------------------
REBALANCE = {"mean_reversion": 1,  # daily: a one-week signal fades within a few days
             "momentum": 21,       # monthly: the 12-1 month return moves slowly
             "quality": 21,        # monthly: fundamentals change once a quarter
             "value": 21}          # monthly: same, only the price moves between two filings

# --- Diagnostics, improvement, robustness ---------------------------------------------------
IC_HORIZONS = (1, 5, 21, 63)       # a day, a week, a month, a quarter: where each signal lives
DECILES = 10                       # fine enough to see whether the relation is monotonic
DECILE_HORIZON = 21                # one month, the holding period of the slow signals
MR_REBALANCE_GRID = range(1, 11)   # from daily to every two weeks for mean-reversion
MR_REBALANCE_IMPROVED = 4          # the frequency kept after the grid (§9): the only positive one at 2 bp
SUBPERIODS = {"2011-2015": ("2011", "2015"),   # post-crisis, zero rates
              "2016-2019": ("2016", "2019"),   # late cycle, growth stocks dominate
              "2020-2024": ("2020", "2024")}   # Covid, inflation, rising rates
COST_GRID = np.arange(0.0, 21.0, 1.0)  # from 0 to 20 bp, to read each book's break-even
MONITOR_WINDOW = 63                # a rolling quarter for the monitoring indicators

pd.set_option("display.width", 160)

In [ ]:
frame = pd.read_parquet(CLEAN_PATH)
pd.Series({"rows": f"{len(frame):,}", "companies": frame["gvkey"].nunique(),
           "period": f"{frame['date'].min():%Y-%m-%d} → {frame['date'].max():%Y-%m-%d}"},
          name="panel").to_frame()

## 3. Data exploration

Before designing anything, a look at what the panel contains: which sectors make up the
universe, how daily returns are distributed, and where the fundamentals are missing. Only member
rows are used, and only aggregates are shown.

In [ ]:
members = frame.loc[frame["in_universe"]]
sector_mix = (members.groupby([members["date"].dt.year, "gics_sector"])["gvkey"].nunique()
                     .unstack().fillna(0))

ax = sector_mix.div(sector_mix.sum(axis=1), axis=0).plot.area(figsize=(10, 3.5), lw=0)
ax.set(title="sector composition of the universe (share of members)", xlabel="", ylim=(0, 1))
ax.legend(ncol=3, fontsize=7, loc="lower left")
plt.show()

Distribution of daily price returns on member days. The tails tell us how much an extreme value
could weigh in a signal, and whether some "returns" are really data artefacts (§4).

In [ ]:
members["ret_1d"].describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999]).to_frame("ret_1d (members)").round(4)

Missing rate on member rows, field by field.

In [ ]:
raw_columns = ["adj_close", "ret_1d", "trfd", "shares", *FUNDAMENTAL_FIELDS]
missing_by_field = members[raw_columns].isna().mean().sort_values(ascending=False)
missing_by_field.map("{:.1%}".format).to_frame("missing rate (members)")

Then by month, for the fields the signals use. A gap concentrated on one period would be a data
problem. A stable level reflects how companies report (a bank does not publish current assets).

In [ ]:
signal_fields = ["CEQQ", "NIQ", "DLTTQ", "DLCQ", "ACTQ", "LCTQ", "ATQ"]
missing_by_month = members.groupby(members["date"].dt.to_period("M"))[signal_fields] \
                          .apply(lambda g: g.isna().mean())

fig, ax = plt.subplots(figsize=(10, 3))
missing_by_month.plot(ax=ax, lw=1)
ax.set(title="missing rate per month, universe members", xlabel="", ylabel="share missing")
ax.legend(ncol=7, fontsize=8)
plt.show()

## 4. Data loopholes

Five traps in this data, each of which returns a **plausible number instead of an error**. For
each one: how big it is on this panel, and how it is closed. (The sixth trap, using a quarter
before it was filed, is closed in `01_data.ipynb`.)

### 4.1 Splits

On a split day the raw close jumps (−67% for a 3-for-1 split). `adj_close = close / adj_factor`
must absorb it, otherwise every split prints as a crash and a huge fake reversal.

In [ ]:
ordered = frame.sort_values(["gvkey", "date"])
by_company = ordered.groupby("gvkey")
previous_factor = by_company["adj_factor"].shift()
split_day = (ordered["adj_factor"].ne(previous_factor) & previous_factor.notna()
             & ordered["iid"].eq(by_company["iid"].shift()))
raw_move = ordered["close"] / by_company["close"].shift() - 1
adjusted_move = ordered["adj_close"] / by_company["adj_close"].shift() - 1

pd.Series({
    "days where adj_factor changes": int(split_day.sum()),
    "median |raw return| on those days": f"{raw_move[split_day].abs().median():.1%}",
    "median |adjusted return| on those days": f"{adjusted_move[split_day].abs().median():.1%}",
}, name="splits").to_frame()

### 4.2 Dividends and spin-offs

`adj_factor` handles splits but not distributions: on a spin-off day the price return shows a
large fall that shareholders never lost, and dividends (≈ 2.5%/yr here) are missing altogether.
The fix is the total return: the daily change of `adj_close × trfd`, where `trfd` is the
vendor's cumulative total-return factor. As for `ret_1d`, no return across a change of issue or a
gap of more than `MAX_GAP_DAYS` days. This is the return the simulator earns.

In [ ]:
def total_return(df: pd.DataFrame) -> pd.Series:
    """Daily change of adj_close × trfd, per company."""
    ordered = df.sort_values(["gvkey", "date"], kind="stable")
    company = ordered["gvkey"]
    index = ordered["adj_close"] * ordered["trfd"]
    ret = index / index.groupby(company).shift() - 1

    valid = ordered["date"].groupby(company).diff().dt.days <= MAX_GAP_DAYS
    valid &= ordered["iid"].eq(ordered["iid"].groupby(company).shift())
    return ret.where(valid).reindex(df.index)

We compute it, then count the extreme member days before and after the fix, and check that the
total return is never below the price return (`trfd` only ever adds distributions).

In [ ]:
frame["ret_1d_total"] = total_return(frame)
member_rows = frame["in_universe"]
extreme_price = member_rows & (frame["ret_1d"].abs() > EXTREME_RETURN)
fixed_by_trfd = extreme_price & (frame["ret_1d_total"].abs() <= EXTREME_RETURN)
extreme_total = member_rows & (frame["ret_1d_total"].abs() > EXTREME_RETURN)
both = frame[["ret_1d", "ret_1d_total"]].dropna()
total_below_price = (both["ret_1d_total"] < both["ret_1d"] - 1e-9).mean()

pd.Series({
    f"member days with |ret_1d| > {EXTREME_RETURN:.0%}": int(extreme_price.sum()),
    "of which brought below the threshold by trfd (distributions)": int(fixed_by_trfd.sum()),
    f"member days with |ret_1d_total| > {EXTREME_RETURN:.0%}": int(extreme_total.sum()),
    "share of days where total return < price return": f"{total_below_price:.3%}",
}, name="distributions").to_frame()

### 4.3 Frozen prices

Several identical closes in a row point to a suspended stock or stale data: the stock would show
a zero return and look "safe". Each series is split into runs of consecutive identical closes,
and a run of at least `FROZEN_DAYS` days is flagged.

In [ ]:
def frozen_runs(df: pd.DataFrame, min_days: int = FROZEN_DAYS) -> pd.Series:
    """True on rows that belong to a run of at least `min_days` identical closes."""
    ordered = df.sort_values(["gvkey", "date"])
    same = ordered["close"].eq(ordered.groupby("gvkey")["close"].shift())
    run_id = (~same).cumsum()
    length = ordered.groupby(run_id)["close"].transform("size")
    return (length >= min_days).reindex(df.index)

In [ ]:
frozen = frozen_runs(frame) & frame["in_universe"]
pd.Series({f"member days in a run of ≥ {FROZEN_DAYS} identical closes": int(frozen.sum()),
           "companies affected": frame.loc[frozen, "gvkey"].nunique()},
          name="frozen prices").to_frame()

### 4.4 Negative book equity

When common equity `CEQQ` is negative, every ratio built on it flips sign: a loss over negative
equity reads as a **positive** return on equity, and a negative book-to-market as "expensive"
for the wrong reason. These rows are masked in the feature design (§5).

In [ ]:
negative_book = members["CEQQ"] <= 0
pd.Series({"member days with CEQQ ≤ 0": f"{negative_book.mean():.2%}",
           "companies with at least one CEQQ ≤ 0": members.loc[negative_book, "gvkey"].nunique()},
          name="negative book equity").to_frame()

### 4.5 Share count: issue versus company

The dataset carries two share counts: `shares` (`CSHOC`), this issue's daily count, and `CSHOQ`,
the company's count in millions across **every share class**. Multiplying one issue's price by
every class's shares gives dual-class companies a market cap they do not have. We measure how
often the two disagree by more than 20%; the market cap in §5 uses `shares`.

In [ ]:
share_ratio = (members["CSHOQ"] * 1e6 / members["shares"]).where(members["shares"] > 0)
disagree = (share_ratio - 1).abs() > 0.2
pd.Series({"median CSHOQ / shares": round(share_ratio.median(), 3),
           "member days where they differ by > 20%": f"{disagree.mean():.1%}",
           "companies concerned": members.loc[disagree, "gvkey"].nunique()},
          name="share count").to_frame()

### Summary of the loopholes

In [ ]:
pd.DataFrame([
    ("splits", f"{int(split_day.sum())} split days", "adj_close = close / adj_factor",
     f"median |adjusted return| on split days: {adjusted_move[split_day].abs().median():.1%}"),
    ("dividends, spin-offs", f"{int(extreme_price.sum())} member days with |ret_1d| > {EXTREME_RETURN:.0%}",
     "total return adj_close × trfd", f"{int(extreme_total.sum())} left after the fix"),
    ("frozen prices", f"{int(frozen.sum())} member days", "flagged; no impact on ranks if rare", "§4.3"),
    ("negative book equity", f"{negative_book.mean():.2%} of member days", "masked in every equity ratio", "§5"),
    ("share count", f"{disagree.mean():.1%} of member days", "market cap on the issue's own shares", "§5"),
], columns=["loophole", "size on this panel", "fix", "after the fix"]).set_index("loophole")

## 5. Feature design

The dataset ships what Compustat filed, not ratios. This section turns the raw fields into the
quantities the signals need, and closes the loopholes above along the way.

| feature | definition | used by |
|---|---|---|
| `market_cap` | close × shares (this issue's count) | value |
| `book_to_market` | CEQQ / market_cap, masked when CEQQ ≤ 0 | value |
| `pe_proxy` | 4 quarters of NIQ / market_cap (earnings yield) | value |
| `return_on_equity` | 4 quarters of NIQ / CEQQ, masked when CEQQ ≤ 0 | quality |
| `leverage` | (DLTTQ + DLCQ) / CEQQ, masked when CEQQ ≤ 0 | quality |
| `working_capital` | (ACTQ − LCTQ) / ATQ | quality |
| `past_return` | P_{t-short} / P_{t-long} − 1 on the total-return index | mean-reversion, momentum |

**Treatment of missing values: nothing is filled in.** A missing value is not a zero. Net income is
summed over **4 adjacent quarters** (60 to 120 days apart), so a missing quarter gives NaN rather
than a sum over 3 quarters. The composite signals (§6) average **the ranks that exist**
(`nanmean`), so a bank with no `working_capital` is still scored on its other legs.

Each function is taken from the project's `features.py`. `trailing` first: the sum of the last
`quarters` quarters published as of each row's date.

In [ ]:
def trailing(df: pd.DataFrame, field: str, quarters: int = QUARTERS) -> pd.Series:
    """Sum of the last `quarters` quarters of `field` in force on each row.

    Quarters are the distinct `datadate`s already published, so the sum only uses what was public
    on the row's date. NaN until `quarters` quarters exist, and unless their ends are 60 to 120
    days apart.
    """
    q = (df.loc[df["datadate"].notna(), ["gvkey", "datadate", field]]
           .drop_duplicates(["gvkey", "datadate"])
           .sort_values(["gvkey", "datadate"]))
    by = q.groupby("gvkey", sort=False)

    # min_periods = window: a NaN inside the window leaves too few values, so NaN.
    total = by[field].transform(lambda s: s.rolling(quarters, min_periods=quarters).sum())

    if quarters > 1:
        gap = by["datadate"].diff().dt.days
        adjacent = gap.between(*ADJACENT_DAYS).astype(float)
        ok = (adjacent.groupby(q["gvkey"], sort=False)
                      .transform(lambda s: s.rolling(quarters - 1).min()) == 1)
        total = total.where(ok)

    per_quarter = pd.Series(total.to_numpy(),
                            index=pd.MultiIndex.from_frame(q[["gvkey", "datadate"]]))
    rows = pd.MultiIndex.from_frame(df[["gvkey", "datadate"]])
    return pd.Series(per_quarter.reindex(rows).to_numpy(), index=df.index, name=field)

Market cap: the **raw** close × the share count **of this issue**, in millions of dollars (§4.5).

In [ ]:
def market_cap(df: pd.DataFrame) -> pd.Series:
    """close × shares, in $m; NaN when the share count is not positive."""
    return (df["close"] * df["shares"] / 1e6).where(df["shares"] > 0)

Book-to-market: common equity over market cap, masked when `CEQQ ≤ 0` (§4.4). The higher it is,
the cheaper the stock.

In [ ]:
def book_to_market(df: pd.DataFrame) -> pd.Series:
    """CEQQ / market_cap, masked where CEQQ ≤ 0."""
    return (df["CEQQ"] / market_cap(df)).where(df["CEQQ"] > 0)

Earnings yield (column `pe_proxy`): four quarters of net income divided by market cap. It is the
inverse of a P/E ratio, so the higher it is, the cheaper the stock. Four quarters, because a
single quarter carries its season.

In [ ]:
def earnings_yield(df: pd.DataFrame) -> pd.Series:
    """Four quarters of NIQ / market_cap."""
    return trailing(df, "NIQ") / market_cap(df)

Return on equity: four quarters of net income divided by `CEQQ`, masked when `CEQQ ≤ 0`.

In [ ]:
def return_on_equity(df: pd.DataFrame) -> pd.Series:
    """Four quarters of NIQ / CEQQ, masked where CEQQ ≤ 0."""
    return (trailing(df, "NIQ") / df["CEQQ"]).where(df["CEQQ"] > 0)

Leverage: debt over **equity** (not over assets), masked when `CEQQ ≤ 0`. A company with no debt
has a leverage of 0, the best score.

In [ ]:
def leverage(df: pd.DataFrame) -> pd.Series:
    """(DLTTQ + DLCQ) / CEQQ, masked where CEQQ ≤ 0."""
    return ((df["DLTTQ"] + df["DLCQ"]) / df["CEQQ"]).where(df["CEQQ"] > 0)

Working capital over total assets. It is NaN for most financials, which do not report current
assets.

In [ ]:
def working_capital(df: pd.DataFrame) -> pd.Series:
    """(ACTQ − LCTQ) / ATQ."""
    return (df["ACTQ"] - df["LCTQ"]) / df["ATQ"]

`build_features` adds every fundamental feature to the panel.

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Every fundamental feature, added to the panel."""
    df["market_cap"] = market_cap(df)
    df["book_to_market"] = book_to_market(df)
    df["pe_proxy"] = earnings_yield(df)
    df["return_on_equity"] = return_on_equity(df)
    df["leverage"] = leverage(df)
    df["working_capital"] = working_capital(df)
    return df

In [ ]:
frame = build_features(frame)
members = frame.loc[frame["in_universe"]]

FEATURES = ["return_on_equity", "leverage", "working_capital", "book_to_market", "pe_proxy"]
members[FEATURES].isna().mean().map("{:.1%}".format).to_frame("missing rate after treatment")

The missing `working_capital` should come almost entirely from financials. This is why the
quality composite averages the legs that exist rather than excluding these companies.

In [ ]:
members.groupby("gics_sector")["working_capital"] \
       .apply(lambda s: s.isna().mean()).sort_values(ascending=False) \
       .map("{:.0%}".format).to_frame("working_capital missing")

Distribution of the features on members. The tails are fat (ROE in the hundreds when equity is
close to zero, for instance). **We do not winsorize**: every signal relies only on
cross-sectional ranks, which an extreme value cannot distort.

In [ ]:
members[FEATURES].describe(percentiles=[0.01, 0.5, 0.99]).T.round(3)

The price features: `past_return` reads the total-return index `adj_close × trfd`, so that
neither a split nor an ex-dividend day is mistaken for a price move (§4.1, §4.2). Lags count the
company's rows and only use prices up to close t.

In [ ]:
def past_return(df: pd.DataFrame, longest: int, shortest: int = 0) -> pd.Series:
    """P_{t-shortest} / P_{t-longest} - 1 on the total-return index, per company."""
    ordered = df.sort_values(["gvkey", "date"])
    price = ordered["adj_close"] * ordered["trfd"]
    by = price.groupby(ordered["gvkey"])
    return (by.shift(shortest) / by.shift(longest) - 1).reindex(df.index)

**Coverage.** A stock is usable by a signal on a date if it has at least one input: the 5-day
return for mean-reversion, the t-252 → t-21 return for momentum (one year of history), one of
ROE / leverage / working capital for quality, one of book-to-market / earnings yield for value.

In [ ]:
usable = pd.DataFrame({
    "mean_reversion": past_return(frame, REVERSAL_LOOKBACK).notna(),
    "momentum": past_return(frame, MOMENTUM_LOOKBACK, MOMENTUM_SKIP).notna(),
    "quality": frame[["return_on_equity", "leverage", "working_capital"]].notna().any(axis=1),
    "value": frame[["book_to_market", "pe_proxy"]].notna().any(axis=1),
}).loc[frame["in_universe"]]
per_day = members.groupby("date").size()
usable_count = usable.groupby(members["date"]).sum()

fig, ax = plt.subplots(figsize=(10, 3))
usable_count.plot(ax=ax, lw=1)
per_day.plot(ax=ax, lw=1, color="0.6", ls="--", label="members")
ax.set(title="usable stocks per date and per signal", xlabel="", ylim=(0, per_day.max() * 1.05))
ax.legend(ncol=5, fontsize=8)
plt.show()

pd.DataFrame({"minimum (stocks)": usable_count.min(), "median (stocks)": usable_count.median(),
              "median share of members": usable_count.div(per_day, axis=0).median().map("{:.1%}".format)})

## 6. The four signals

Every signal has the same shape: **a rank centred on zero among the universe members on that
date**, in [-0.5, 0.5], NaN outside the universe. The higher the score, the more we want to be
long. The `cross_rank` function produces this shape. The price signals use `past_return`
(§5) on the total-return index.

In [ ]:
def cross_rank(df: pd.DataFrame, values: pd.Series) -> pd.Series:
    """`values` ranked among the date's members, centred on zero, in [-0.5, 0.5]."""
    members = values.where(df["in_universe"].astype(bool))
    by = members.groupby(df["date"])
    n = by.transform("count")
    return (by.rank() - (n + 1) / 2) / n

### Mean-reversion

**Intuition.** In the very short term the market overreacts: liquidity flows and forced selling
push prices too far, and then they come back. So we buy the week's losers and sell its winners.
The whole signal is in the minus sign.

$$mr = \text{rank}\big(-(P_t / P_{t-5} - 1)\big)$$

In [ ]:
def mean_reversion(df: pd.DataFrame) -> pd.Series:
    """-(P_t / P_{t-5} - 1), ranked among the date's members."""
    return cross_rank(df, -past_return(df, REVERSAL_LOOKBACK))

### Momentum

**Intuition.** Information spreads slowly: underreaction to news, then herding. A stock that has
risen for a year tends to keep rising. We skip the last month because reversal dominates at that
horizon.

$$mo = \text{rank}\big(P_{t-21} / P_{t-252} - 1\big)$$

In [ ]:
def momentum(df: pd.DataFrame) -> pd.Series:
    """P_{t-21} / P_{t-252} - 1, ranked among the date's members."""
    return cross_rank(df, past_return(df, MOMENTUM_LOOKBACK, MOMENTUM_SKIP))

### Quality

**Intuition.** The market undervalues profitable, low-leverage firms: they are less risky than it
thinks and disappoint less often. The score is the average of three ranks. `nanmean` averages the
ranks that exist: a bank with no working capital is still scored on the other two. A company with
no debt has a leverage of 0 and ranks first on that leg.

$$qu = \text{rank}\big(\text{nanmean}[\text{rank}(roe),\ -\text{rank}(lev),\ \text{rank}(wc)]\big)$$

In [ ]:
def quality(df: pd.DataFrame) -> pd.Series:
    """Average of the available ranks of ROE, -leverage and working capital, re-ranked."""
    legs = pd.concat([cross_rank(df, df["return_on_equity"]),
                      -cross_rank(df, df["leverage"]),
                      cross_rank(df, df["working_capital"])], axis=1)
    return cross_rank(df, legs.mean(axis=1))      # mean skips NaN: this is the nanmean

### Value

**Intuition.** Stocks that are cheap relative to their fundamentals earn more, either as
compensation for risk or because the market over-extrapolates bad news. Two measures of
"cheap": book-to-market and earnings yield. Book-to-market is already masked when equity is
negative.

$$va = \text{rank}\big(\text{nanmean}[\text{rank}(b/m),\ \text{rank}(ey)]\big)$$

In [ ]:
def value(df: pd.DataFrame) -> pd.Series:
    """Average of the available ranks of book-to-market and earnings yield, re-ranked."""
    legs = pd.concat([cross_rank(df, df["book_to_market"]),
                      cross_rank(df, df["pe_proxy"])], axis=1)
    return cross_rank(df, legs.mean(axis=1))

We compute the four scores. By construction each must have a zero mean and stay within
[-0.5, 0.5].

In [ ]:
SIGNAL_FUNCTIONS = {"mean_reversion": mean_reversion, "momentum": momentum,
                    "quality": quality, "value": value}
for name, compute in SIGNAL_FUNCTIONS.items():
    frame[name] = compute(frame)
frame.loc[frame["in_universe"], SIGNALS].describe().T.round(3)

### Diagnostics: do the signals predict?

Before building any portfolio, we measure the **IC** (information coefficient): the rank
correlation, date by date, between the previous day's score and the return that follows. An IC of
0.02 is a real signal; 0.2 almost always means the future was used.

Two simulator tools are needed first: reshaping the panel to date × company, and building the
forward return. The return is dated on the day the position is **held**: the value at t is earned
from t to t+h. It is the only place a forward return is built, so it cannot leak into a signal.

In [ ]:
def _wide(frame: pd.DataFrame, column: str) -> pd.DataFrame:
    """One column of the panel as date × gvkey (strict pivot: a duplicate would raise)."""
    return frame.pivot(index="date", columns="gvkey", values=column).sort_index()

In [ ]:
def forward_return(frame: pd.DataFrame, horizon: int = 1) -> pd.DataFrame:
    """r_{i, t→t+h}, date × gvkey, compounded from the daily total return."""
    column = TOTAL_RETURN
    if column not in frame.columns:
        warnings.warn(f"{column} missing: using {FALLBACK}, without dividends.", stacklevel=2)
        column = FALLBACK
    ret = _wide(frame, column)
    log_ret = np.log1p(ret)              # compound through a sum rather than a product
    fwd = log_ret.shift(-1)              # on row t, the return of t+1
    fwd_sum = fwd[::-1].rolling(horizon, min_periods=horizon).sum()[::-1]
    return np.expm1(fwd_sum)

In [ ]:
def ic(frame: pd.DataFrame, signal: str, *, horizon: int = 21, lag: int = SIGNAL_LAG) -> pd.Series:
    """Spearman correlation, per date, between the score lagged by `lag` and the `horizon`-day return."""
    score = _wide(frame, signal).shift(lag)
    fwd = forward_return(frame, horizon=horizon)
    score, fwd = score.align(fwd, join="inner")
    return score.corrwith(fwd, axis=1, method="spearman")

In [ ]:
def decay(frame: pd.DataFrame, signal: str, horizons=IC_HORIZONS, *, lag: int = SIGNAL_LAG) -> pd.DataFrame:
    """Mean IC per horizon: flat = slow and cheap to trade; alive only at 1 day = paying to trade every day."""
    rows = {}
    for h in horizons:
        series = ic(frame, signal, horizon=h, lag=lag)
        rows[h] = {"mean_ic": series.mean(), "std_ic": series.std(), "n_dates": series.count()}
    return pd.DataFrame(rows).T

In [ ]:
ic_decay = pd.DataFrame({name: decay(frame, name)["mean_ic"] for name in SIGNALS})
ic_decay.index.name = "horizon (days)"

ax = ic_decay.plot.bar(figsize=(10, 3), rot=0, title="mean IC by horizon")
ax.axhline(0, color="0.5", lw=0.8)
plt.show()
ic_decay.round(4)

**Deciles** complement the IC. Each day, stocks are sorted into 10 groups by the previous day's
score, and we look at the average return over the following month. A good signal gives a steady
slope from decile 1 to decile 10. A slope that only comes from one extreme is more fragile.

In [ ]:
def decile_returns(frame: pd.DataFrame, signal: str, *, horizon: int = DECILE_HORIZON,
                   n: int = DECILES, lag: int = SIGNAL_LAG) -> pd.Series:
    """Mean `horizon`-day forward return per decile of the lagged score, averaged over dates."""
    score = _wide(frame, signal).shift(lag)
    fwd = forward_return(frame, horizon=horizon)
    score, fwd = score.align(fwd, join="inner")
    bucket = np.ceil(score.rank(axis=1, pct=True) * n).clip(1, n)
    long = pd.DataFrame({"decile": bucket.stack(), "fwd": fwd.stack()}).dropna()
    per_date = long.groupby([long.index.get_level_values(0), "decile"])["fwd"].mean()
    return per_date.groupby(level="decile").mean()

In [ ]:
deciles = pd.DataFrame({name: decile_returns(frame, name) for name in SIGNALS}) * 1e4
deciles.index = deciles.index.astype(int)
deciles.index.name = "decile"

fig, axes = plt.subplots(1, 4, figsize=(13, 2.8), sharey=True)
for ax, name in zip(axes, SIGNALS):
    deciles[name].plot.bar(ax=ax, rot=0, color="tab:blue")
    ax.set(title=name, xlabel="decile")
axes[0].set_ylabel(f"mean {DECILE_HORIZON}-day return (bp)")
plt.tight_layout()
plt.show()

spread = (deciles.loc[DECILES] - deciles.loc[1]).rename(f"D{DECILES} − D1 (bp over {DECILE_HORIZON} d)")
spread.round(1).to_frame()

**Are they different bets?** Rank correlation between the four scores, computed date by date and
then averaged. Low correlations mean the blend genuinely diversifies.

In [ ]:
per_date_corr = frame.loc[frame["in_universe"]].groupby("date")[SIGNALS].corr(method="spearman")
per_date_corr.groupby(level=1).mean().loc[SIGNALS, SIGNALS].round(2)

## 7. The simulator

Five rules, the same for every team:

1. **Book.** Equal-weighted quintile long-short: long the top 20%, short the bottom 20%.
   Dollar-neutral, gross exposure 1 (0.5 per leg).
2. **Combine.** Weight 1/N per signal. We average the **portfolios**, not the P&Ls: two signals
   that disagree cancel out before paying any cost.
3. **Risk.** Target 10% annualised volatility by multiplying the portfolio by 10%/σ. σ is the
   63-day volatility of the portfolio's P&L, as known the day before.
4. **Execution.** A signal computed at close t is executed at close t+1 and earns the return
   from t+1 to t+2.
5. **Cost.** Linear: c × Σ|Δw| every day, with c = 2, 5 or 10 bp. No ½: every leg pays.

Each function is taken from the project's `simulator.py`.

**Rule 1.** `weights` builds the portfolio. `rebalance=n` re-forms it every n days and **holds**
it in between. Eligibility is checked at each re-formation.

In [ ]:
def weights(frame: pd.DataFrame, signal: str, *, quantiles: int = QUANTILES,
            lag: int = SIGNAL_LAG, rebalance: int = 1) -> pd.DataFrame:
    """Long the top quantile, short the bottom one, 0.5 gross per leg, re-formed every `rebalance` days."""
    sig_wide = frame.pivot(index="date", columns="gvkey", values=signal).sort_index()
    # NaN (stock absent that day) → False, once and for all
    universe_wide = frame.pivot(index="date", columns="gvkey", values="in_universe").sort_index().eq(True)

    lagged = sig_wide.shift(lag)
    dates = sig_wide.index

    book = pd.DataFrame(0.0, index=dates, columns=sig_wide.columns)
    current = pd.Series(0.0, index=sig_wide.columns)

    for i, d in enumerate(dates):
        if i % rebalance == 0:
            eligible = universe_wide.loc[d]
            # before `lag` days of history exist, we cannot lag
            source = lagged.loc[d] if i >= lag else sig_wide.loc[d]
            scores = source.dropna()
            scores = scores[eligible.reindex(scores.index).fillna(False)]
            current = pd.Series(0.0, index=sig_wide.columns)
            if len(scores) >= quantiles:
                buckets = pd.qcut(scores, quantiles, labels=False, duplicates="drop")
                top = scores[buckets == buckets.max()].index
                bottom = scores[buckets == buckets.min()].index
                current[top] = 0.5 / len(top)
                current[bottom] = -0.5 / len(bottom)
        book.loc[d] = current

    return book

In [ ]:
books = {name: weights(frame, name, rebalance=REBALANCE[name]) for name in SIGNALS}

last_day = books["quality"].index[-1]
w = books["quality"].loc[last_day]
pd.Series({"longs": int((w > 0).sum()), "shorts": int((w < 0).sum()),
           "net exposure": round(w.sum(), 6), "gross exposure": round(w.abs().sum(), 6)},
          name=f"quality book on {last_day:%Y-%m-%d}").to_frame()

**Rule 2.** `combine` averages the portfolios. A signal with no portfolio yet on a date counts as
an empty one.

In [ ]:
def combine(books) -> pd.DataFrame:
    """The average portfolio, weight 1/N per signal."""
    books = list(books)
    n = len(books)
    all_dates = books[0].index
    all_cols = books[0].columns
    for b in books[1:]:
        all_dates = all_dates.union(b.index)
        all_cols = all_cols.union(b.columns)
    total = pd.DataFrame(0.0, index=all_dates, columns=all_cols)
    for b in books:
        total = total.add(b.reindex(index=all_dates, columns=all_cols, fill_value=0.0),
                          fill_value=0.0)
    return total / n

`gross` computes the P&L before costs: the portfolio held at t earns the return from t to t+1. A
position held on a stock that has no return the next day (acquisition, delisting) is marked at
zero, since there is no data on the exit price, and a warning reports the number of cases.

In [ ]:
def gross(book: pd.DataFrame, frame: pd.DataFrame, *, strict: bool = False) -> pd.Series:
    """Σ_i w_{i,t} r_{i,t→t+1}: the portfolio held at t earns the next day."""
    fwd = forward_return(frame, horizon=1)
    book, fwd = book.align(fwd, join="left")

    # the panel's last date has no next day: it is not an unpriced position
    last_date = frame["date"].max()
    fwd_trimmed = fwd.drop(index=last_date, errors="ignore")
    book_trimmed = book.drop(index=last_date, errors="ignore")

    exposed = (book_trimmed != 0) & fwd_trimmed.isna()
    n_unpriced = exposed.sum().sum()
    if n_unpriced > 0:
        first_date = book_trimmed.index[exposed.any(axis=1)][0]
        first_gvkey = exposed.loc[first_date][exposed.loc[first_date]].index[0]
        if strict:
            raise ValueError(f"{n_unpriced} unpriced exposed rows, first: "
                             f"{first_date:%Y-%m-%d} gvkey={first_gvkey}")
        warnings.warn(f"{n_unpriced} exposed rows have no forward return (e.g. delisting), "
                      f"first: {first_date:%Y-%m-%d} — marked flat.", stacklevel=2)

    fwd_filled = fwd_trimmed.where(~exposed, 0.0)
    return (book_trimmed * fwd_filled.fillna(0.0)).sum(axis=1)

We measure the size of these unpriced positions once on the blend, then silence the warning,
which would otherwise repeat on every call.

In [ ]:
blend = combine(books.values())
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    gross(blend, frame)
unpriced = [str(c.message) for c in caught if "exposed rows" in str(c.message)]
exposure_days = int((blend != 0).sum().sum())
pd.Series({"blend position-days": f"{exposure_days:,}",
           "warning": unpriced[0] if unpriced else "no unpriced position"},
          name="unpriced positions").to_frame()

In [ ]:
warnings.filterwarnings("ignore", message=".*exposed rows have no forward return.*")

**Rule 3.** `vol_target` multiplies the portfolio by 10%/σ, where σ is the 63-day volatility of
the portfolio's gross P&L. The lag is **two rows**: the P&L dated t is earned from close t to
close t+1, so the last σ that does not contain the close we trade at ends at t−1. Dates without
63 days of history are not traded.

In [ ]:
def vol_target(book: pd.DataFrame, frame: pd.DataFrame, *, target: float = TARGET_VOL,
               window: int = VOL_WINDOW, max_leverage: float = MAX_LEVERAGE) -> pd.DataFrame:
    """w_t × σ*/σ̂_{t-1}: the portfolio scaled to `target` annualised volatility."""
    pnl = gross(book, frame)
    vol = pnl.rolling(window, min_periods=window).std() * np.sqrt(YEAR)
    vol_known = vol.shift(2)
    factor = (target / vol_known).clip(upper=max_leverage)
    factor = factor.reindex(book.index)
    return book.mul(factor, axis=0)

**Rule 5.** Costs. `traded` gives the notional traded per stock (on the first day, the whole
portfolio is bought), `turnover` sums it, and `cost` multiplies it by the unit cost.

In [ ]:
def traded(book: pd.DataFrame) -> pd.DataFrame:
    """Notional traded per stock and per date, as a fraction of the portfolio."""
    return book.diff().fillna(book).abs()

In [ ]:
def turnover(book: pd.DataFrame) -> pd.Series:
    """Σ_i |w_{i,t} − w_{i,t−1}|: both legs, no ½."""
    return traded(book).sum(axis=1)

In [ ]:
def cost(book: pd.DataFrame, bps: float = DEFAULT_BPS) -> pd.Series:
    """c × Σ_i |Δw_i| per date, in return units; `bps` is c in basis points."""
    return turnover(book) * (bps / 1e4)

In [ ]:
def net(book: pd.DataFrame, frame: pd.DataFrame, bps: float = DEFAULT_BPS) -> pd.Series:
    """Gross P&L minus the cost of getting into it. This is the result."""
    g = gross(book, frame)
    c = cost(book, bps=bps).reindex(g.index).fillna(0.0)
    return g - c

Performance measures. They are only valid on **non-overlapping** daily returns. The break-even
(`breakeven`) is the cost per unit traded that brings the return to zero: a break-even below 2 bp
means the signal cannot be traded.

In [ ]:
def performance(returns: pd.Series, *, periods_per_year: int = YEAR) -> pd.Series:
    """Annualised return and volatility, Sharpe, maximum drawdown, share of winning days."""
    returns = returns.dropna()
    ann_return = returns.mean() * periods_per_year
    ann_vol = returns.std() * np.sqrt(periods_per_year)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    wealth = (1 + returns).cumprod()
    running_max = wealth.cummax().clip(lower=1.0)     # the initial capital counts as a peak
    drawdown = (wealth / running_max - 1).min()
    return pd.Series({"ann_return": ann_return, "ann_vol": ann_vol, "sharpe": sharpe,
                      "max_drawdown": drawdown, "hit_rate": (returns > 0).mean()})

In [ ]:
def breakeven(book: pd.DataFrame, frame: pd.DataFrame) -> float:
    """10⁴ × mean gross return / mean turnover: the cost (in bp) at which the portfolio dies."""
    g = gross(book, frame)
    t = turnover(book).reindex(g.index).fillna(0.0)
    return 1e4 * g.mean() / t.mean()

In [ ]:
def equity_curve(returns: pd.Series, start: float = 100.0) -> pd.Series:
    """Growth of `start`, compounded daily."""
    return start * (1 + returns.fillna(0)).cumprod()

In [ ]:
def report(book: pd.DataFrame, frame: pd.DataFrame, *, bps=COST_BPS) -> pd.Series:
    """One row: gross return, volatility, net Sharpe at each cost level, turnover, break-even."""
    perf_gross = performance(gross(book, frame))
    row = {"gross_ann_return": perf_gross["ann_return"], "ann_vol": perf_gross["ann_vol"]}
    for b in bps:
        row[f"net_{int(b)}bp_sharpe"] = performance(net(book, frame, bps=b))["sharpe"]
    row["turnover_x_yr"] = turnover(book).mean() * YEAR
    row["breakeven_bp"] = breakeven(book, frame)
    return pd.Series(row)

In [ ]:
def run(frame: pd.DataFrame, signals=SIGNALS, *, rebalance=21,
        target: float = TARGET_VOL, bps=COST_BPS) -> pd.DataFrame:
    """The five rules end to end: one row per signal at `target` vol, plus the 1/N blend."""
    if not isinstance(rebalance, dict):
        rebalance = {s: rebalance for s in signals}
    books = {s: weights(frame, s, rebalance=rebalance.get(s, 21)) for s in signals}
    rows = {s: report(vol_target(b, frame, target=target), frame, bps=bps) for s, b in books.items()}
    blend = combine(books.values())
    rows["1/N blend"] = report(vol_target(blend, frame, target=target), frame, bps=bps)
    return pd.DataFrame(rows).T

**Simulator checks.** First, combining the portfolios rather than the P&Ls should reduce
turnover. Second, the volatility target must hold: the one-year realised volatility of the
targeted blend should stay close to 10%.

In [ ]:
each = pd.concat([turnover(b) for b in books.values()], axis=1).sum(axis=1) / len(books)
pd.Series({"the four portfolios separately / 4": each.mean() * YEAR,
           "the combined portfolio": turnover(blend).mean() * YEAR},
          name="turnover (times per year), before vol targeting").round(1).to_frame()

In [ ]:
scaled_blend = vol_target(blend, frame)
realised = pd.DataFrame({"blend, unscaled": gross(blend, frame),
                         "blend, targeted at 10%": gross(scaled_blend, frame)})
ax = (realised.rolling(YEAR).std() * np.sqrt(YEAR) * 100).plot(
    figsize=(10, 3), lw=1.1, title="realised volatility over a rolling year, %")
ax.axhline(TARGET_VOL * 100, color="0.5", lw=0.8, ls="--")
ax.set_xlabel("")
plt.show()

**Rule 4, the lag test.** We compare the gross Sharpe of mean-reversion across lags between
signal and execution. `lag=-1` trades on tomorrow's signal: this is the classic backtest bug, and
it produces an absurd Sharpe. With `lag ≥ 0`, the figures should stay of the same order.

In [ ]:
pd.Series({lag: performance(gross(weights(frame, "mean_reversion", lag=lag), frame))["sharpe"]
           for lag in (-1, 0, 1, 2)},
          name="mean-reversion gross Sharpe by lag").round(2).to_frame()

## 8. Baseline results, gross and net of costs

The four signals with the baseline formulas, plus the 1/N blend. Mean-reversion rebalanced every
day, the other three every month. Each portfolio is targeted at 10% volatility.

In [ ]:
baseline = run(frame, rebalance=REBALANCE)
baseline.round(2)

Detailed performance net of 5 bp: return, volatility, Sharpe, maximum drawdown and share of
winning days.

In [ ]:
scaled = {name: vol_target(book, frame) for name, book in books.items()}
scaled["1/N blend"] = scaled_blend
net_pnl = {name: net(book, frame, bps=DEFAULT_BPS) for name, book in scaled.items()}
pd.DataFrame({name: performance(r) for name, r in net_pnl.items()}).T.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
for name, r in net_pnl.items():
    equity_curve(r).plot(ax=ax, lw=1.1, label=name, logy=True)
ax.axhline(100, color="0.5", lw=0.8)
ax.legend(ncol=5, fontsize=8)
ax.set(title=f"growth of 100, net of {DEFAULT_BPS:.0f} bp, every portfolio at 10% vol", xlabel="")
plt.show()

Correlation of daily net P&Ls. This, more than the correlation of the scores, tells what the
blend actually diversifies.

In [ ]:
pd.DataFrame(net_pnl).corr().round(2)

## 9. The improvement: rebalancing mean-reversion less often

**Which signal, and why.** Mean-reversion is the only signal whose problem is clear and that has
an obvious lever. Its gross edge exists (positive IC at every horizon, positive gross Sharpe at
any lag ≥ 0). But it trades about 300 times a year, and its break-even is below the lowest cost.
It does not die for lack of prediction: it dies of its costs. And since the 1/N blend averages
portfolios, its turnover contaminates the blend.

Momentum and quality already survive costs. Value has a negative IC at every horizon: its problem
is the signal itself, not the way it is traded.

**The idea.** The signal measures the 5-day return and its IC stays positive at 5 and 21 days.
Holding the portfolio for a few days should therefore cost little gross edge while sharply
reducing the number of trades. We test every frequency from 1 to 10 days.

In [ ]:
mr_books = {rb: vol_target(weights(frame, "mean_reversion", rebalance=rb), frame)
            for rb in MR_REBALANCE_GRID}
mr_grid = pd.DataFrame({rb: report(b, frame) for rb, b in mr_books.items()}).T
mr_grid.index.name = "rebalance (days)"
mr_grid.round(3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7.5))

for rb, b in mr_books.items():
    equity_curve(net(b, frame, bps=DEFAULT_BPS)).plot(ax=axes[0, 0], lw=1.1, label=f"{rb} d")
axes[0, 0].axhline(100, color="0.5", lw=0.8)
axes[0, 0].set(title=f"growth of 100, net of {DEFAULT_BPS:.0f} bp", xlabel="")
axes[0, 0].legend(ncol=5, fontsize=7)

for b in COST_BPS:
    mr_grid[f"net_{int(b)}bp_sharpe"].plot(ax=axes[0, 1], marker="o", lw=1.5, label=f"{int(b)} bp")
axes[0, 1].axhline(0, color="0.5", lw=0.8, ls="--")
axes[0, 1].set(title="net Sharpe by frequency", xlabel="rebalance (days)", ylabel="Sharpe")
axes[0, 1].legend()

mr_grid["turnover_x_yr"].plot.bar(ax=axes[1, 0], color="darkred", rot=0)
axes[1, 0].set(title="annual turnover", xlabel="rebalance (days)", ylabel="times per year")

mr_grid["breakeven_bp"].plot.bar(ax=axes[1, 1], color="darkgreen", rot=0)
for b in COST_BPS:
    axes[1, 1].axhline(b, color="0.4", lw=0.8, ls="--")
axes[1, 1].set(title="break-even (bp); lines: 2, 5 and 10 bp", xlabel="rebalance (days)")

plt.tight_layout()
plt.show()

**The choice.** We keep `MR_REBALANCE_IMPROVED` (4 days): it gave the best net Sharpe at 2 bp in
our first grid, and the only positive one at that cost level. Caution: this parameter was **chosen
by looking at the result over the whole sample**. The curve is not monotonic (3 and 5 days do
worse than 4), so part of the gain is probably noise. We now compare the baseline signal with the
improved one, then the baseline blend with the blend that uses the improved mean-reversion.

In [ ]:
improved_books = {**books, "mean_reversion": weights(frame, "mean_reversion",
                                                     rebalance=MR_REBALANCE_IMPROVED)}
improved_blend = combine(improved_books.values())
scaled_improved_mr = mr_books[MR_REBALANCE_IMPROVED]
scaled_improved_blend = vol_target(improved_blend, frame)

comparison = pd.DataFrame({
    "mean_reversion, baseline (1 d)": baseline.loc["mean_reversion"],
    f"mean_reversion, improved ({MR_REBALANCE_IMPROVED} d)": mr_grid.loc[MR_REBALANCE_IMPROVED],
    "1/N blend, baseline": baseline.loc["1/N blend"],
    "1/N blend, with improved MR": report(scaled_improved_blend, frame),
}).T
comparison.round(3)

**What did not work.** In our first grid (1 to 5 days), the break-even never exceeded 2.9 bp
(reached at 4 days). None of these frequencies makes mean-reversion profitable at the 5 bp
reference cost: the improvement reduces the loss without creating a profit. The grid above
extends the test to 10 days. The last row of the comparison table shows what the blend recovers
when its most expensive signal trades half as much.

## 10. Robustness

### Sub-periods

A 14-year Sharpe can hide a single good period. We split the sample into three market regimes,
net of 5 bp, including the improved mean-reversion and blend.

In [ ]:
net_pnl[f"mean_reversion ({MR_REBALANCE_IMPROVED} d)"] = net(scaled_improved_mr, frame, bps=DEFAULT_BPS)
net_pnl["1/N blend (improved MR)"] = net(scaled_improved_blend, frame, bps=DEFAULT_BPS)

by_period = pd.DataFrame({
    period: {name: performance(r.loc[start:end])["sharpe"] for name, r in net_pnl.items()}
    for period, (start, end) in SUBPERIODS.items()
})
by_period.round(2)

The same net Sharpe, year by year. One bar per year and per portfolio, to see whether the result
comes from only a few years.

In [ ]:
by_year = pd.DataFrame({name: r.groupby(r.index.year).apply(lambda x: performance(x)["sharpe"])
                        for name, r in net_pnl.items()}).dropna(how="all")
ax = by_year[SIGNALS].plot.bar(figsize=(12, 3.2), rot=0, width=0.8,
                               title=f"net Sharpe at {DEFAULT_BPS:.0f} bp, by year")
ax.axhline(0, color="0.5", lw=0.8)
ax.set_xlabel("")
plt.show()
by_year.round(2)

The IC year by year, with its t-stat and the share of days it is positive. Three good years out
of fourteen do not make a signal.

In [ ]:
def ic_by_year(series: pd.Series) -> pd.DataFrame:
    """Mean IC, t-stat and share of positive days, by year."""
    by_year = series.groupby(series.index.year)
    mean_ic = by_year.mean()
    t_stat = mean_ic / (by_year.std() / np.sqrt(by_year.count()))
    return pd.DataFrame({"mean_ic": mean_ic, "t_stat": t_stat,
                         "hit_rate": by_year.apply(lambda s: (s > 0).mean()),
                         "n_days": by_year.count()})

In [ ]:
ic_years = pd.concat({name: ic_by_year(ic(frame, name, horizon=21))["t_stat"] for name in SIGNALS},
                     axis=1).dropna(how="all")
ic_years.index.name = "year"
ic_years.round(2)

### Cost sensitivity

The cost is linear, so the net P&L is `gross − c × turnover` and can be recomputed without
re-simulating. We read the net Sharpe from 0 to 20 bp. Where each curve crosses zero is its
break-even.

In [ ]:
cost_books = {**scaled, f"mean_reversion ({MR_REBALANCE_IMPROVED} d)": scaled_improved_mr,
              "1/N blend (improved MR)": scaled_improved_blend}
cost_curve = {}
for name, book in cost_books.items():
    g = gross(book, frame)
    t = turnover(book).reindex(g.index).fillna(0.0)
    cost_curve[name] = {bp: performance(g - t * bp / 1e4)["sharpe"] for bp in COST_GRID}
cost_curve = pd.DataFrame(cost_curve)
cost_curve.index.name = "cost (bp)"

ax = cost_curve.plot(figsize=(10, 3.5), lw=1.2, title="net Sharpe by cost per unit traded")
ax.axhline(0, color="0.5", lw=0.8)
for b in COST_BPS:
    ax.axvline(b, color="0.8", lw=0.8, ls="--")
ax.set(xlabel="cost (bp)", ylabel="net Sharpe", ylim=(-3, None))
ax.legend(fontsize=7, ncol=2)
plt.show()
cost_curve.loc[[0.0, *COST_BPS, 20.0]].T.round(2)

## 11. Monitoring

`monitoring.py` was empty in the project package: it was an open question of the assignment.
This section gives **our answer**, written for this export. It answers the question: if the blend
(with the improved mean-reversion) went live tomorrow, which numbers would we look at every
morning, and which decision does each one trigger?

Six quantities, each over a rolling quarter (`MONITOR_WINDOW`):

| quantity | what it watches | threshold | decision |
|---|---|---|---|
| realised volatility | the risk model | > 1.5 × target | cut exposure, review σ |
| rolling IC of each signal | signal health | < 0 for 6 months | drop the signal from the blend, investigate |
| cost paid / gross return | real costs | > 100% over a quarter | rebalance less often |
| current drawdown | the loss | < −2 × vol target | cut the portfolio in half |
| net exposure by sector | unintended bets | > 10% of gross on one sector | neutralise by sector |
| stocks scored per signal | the data | < 80% of members | freeze the signal, check the data |

In [ ]:
def realised_vol(pnl: pd.Series, window: int = MONITOR_WINDOW) -> pd.Series:
    """Annualised realised volatility over `window` days."""
    return pnl.rolling(window).std() * np.sqrt(YEAR)

In [ ]:
def rolling_ic(frame: pd.DataFrame, signal: str, window: int = MONITOR_WINDOW, horizon: int = 21) -> pd.Series:
    """Rolling mean IC over `window` dates."""
    return ic(frame, signal, horizon=horizon).rolling(window).mean()

In [ ]:
def cost_ratio(book: pd.DataFrame, frame: pd.DataFrame, bps: float = DEFAULT_BPS,
               window: int = MONITOR_WINDOW) -> pd.Series:
    """Cost paid over the window divided by the window's gross return (NaN if the gross is not positive)."""
    g = gross(book, frame).rolling(window).sum()
    c = cost(book, bps=bps).reindex(g.index).fillna(0.0).rolling(window).sum()
    return (c / g).where(g > 0)

In [ ]:
def drawdown(pnl: pd.Series) -> pd.Series:
    """Current drawdown of the wealth curve."""
    wealth = (1 + pnl.fillna(0)).cumprod()
    return wealth / wealth.cummax().clip(lower=1.0) - 1

In [ ]:
def sector_exposure(book: pd.DataFrame, frame: pd.DataFrame) -> pd.DataFrame:
    """Net exposure by GICS sector, as a fraction of gross exposure, date × sector."""
    sector = frame.drop_duplicates("gvkey", keep="last").set_index("gvkey")["gics_sector"]
    by_sector = book.T.groupby(sector.reindex(book.columns)).sum().T
    return by_sector.div(book.abs().sum(axis=1), axis=0)

In [ ]:
def coverage(frame: pd.DataFrame, signals=SIGNALS) -> pd.DataFrame:
    """Share of universe members with a score, per date and per signal."""
    members = frame.loc[frame["in_universe"]]
    return members[signals].notna().groupby(members["date"]).mean()

The dashboard of the monitored portfolio: the blend with the improved mean-reversion, targeted at
10% volatility.

In [ ]:
live_book = scaled_improved_blend
live_pnl = net(live_book, frame, bps=DEFAULT_BPS)
monitor = {
    "vol": realised_vol(live_pnl),
    "ic": pd.DataFrame({name: rolling_ic(frame, name) for name in SIGNALS}),
    "cost": cost_ratio(live_book, frame),
    "drawdown": drawdown(live_pnl),
    "sectors": sector_exposure(live_book.fillna(0.0), frame),
    "coverage": coverage(frame),
}

fig, axes = plt.subplots(3, 2, figsize=(13, 9))
(monitor["vol"] * 100).plot(ax=axes[0, 0], lw=1)
axes[0, 0].axhline(TARGET_VOL * 100, color="0.5", ls="--", lw=0.8)
axes[0, 0].axhline(TARGET_VOL * 150, color="red", ls=":", lw=0.8)
axes[0, 0].set(title="realised volatility (%)", xlabel="")

monitor["ic"].plot(ax=axes[0, 1], lw=0.8)
axes[0, 1].axhline(0, color="red", ls=":", lw=0.8)
axes[0, 1].set(title="rolling IC (21 d)", xlabel="")
axes[0, 1].legend(fontsize=7, ncol=2)

monitor["cost"].clip(upper=3).plot(ax=axes[1, 0], lw=1)
axes[1, 0].axhline(1, color="red", ls=":", lw=0.8)
axes[1, 0].set(title="cost paid / gross return (capped at 3)", xlabel="")

(monitor["drawdown"] * 100).plot(ax=axes[1, 1], lw=1)
axes[1, 1].axhline(-TARGET_VOL * 200, color="red", ls=":", lw=0.8)
axes[1, 1].set(title="current drawdown (%)", xlabel="")

monitor["sectors"].abs().max(axis=1).plot(ax=axes[2, 0], lw=1)
axes[2, 0].axhline(0.10, color="red", ls=":", lw=0.8)
axes[2, 0].set(title="largest net sector exposure (fraction of gross)", xlabel="")

monitor["coverage"].plot(ax=axes[2, 1], lw=0.8)
axes[2, 1].axhline(0.8, color="red", ls=":", lw=0.8)
axes[2, 1].set(title="share of members scored", xlabel="")
axes[2, 1].legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.show()

The status on the last day of the panel, with the thresholds from the table above.

In [ ]:
last = live_pnl.index[-1]
sectors_last = monitor["sectors"].loc[last]
ic_last = monitor["ic"].dropna(how="all").iloc[-1]   # the 21-day IC is only known 21 days later
status = pd.DataFrame([
    ("realised volatility", f"{monitor['vol'].loc[last]:.1%}", f"> {1.5 * TARGET_VOL:.0%}"),
    (f"weakest rolling IC (on {ic_last.name:%Y-%m-%d})",
     f"{ic_last.min():+.3f} ({ic_last.idxmin()})", "< 0 for 6 months"),
    ("cost / gross return", f"{monitor['cost'].loc[last]:.0%}" if pd.notna(monitor['cost'].loc[last]) else "gross ≤ 0 over the window", "> 100%"),
    ("current drawdown", f"{monitor['drawdown'].loc[last]:.1%}", f"< {-2 * TARGET_VOL:.0%}"),
    ("largest sector exposure", f"{sectors_last.abs().max():.1%} ({sectors_last.abs().idxmax()})", "> 10%"),
    ("lowest coverage", f"{monitor['coverage'].iloc[-1].min():.0%}", "< 80%"),
], columns=["indicator", f"value on {last:%Y-%m-%d}", "alert threshold"]).set_index("indicator")
status

## 12. Analysis and conclusion

*The figures quoted here come from running the pipeline on the MaQI platform (2011-2024 panel,
snapshot `20260711`). They are those of the baseline table in §8 and the grid in §9. For the
sections added in this export (exploration, loopholes, deciles, sub-periods, monitoring), we refer to the cell outputs
rather than quoting values.*

### What the results show

| portfolio | net Sharpe 2 bp | 5 bp | 10 bp | turnover (×/yr) | break-even (bp) |
|---|---|---|---|---|---|
| mean-reversion (1 d) | −0.24 | −1.09 | −2.50 | 305 | 1.2 |
| momentum | 0.39 | 0.35 | 0.29 | 13 | 33.4 |
| quality | 0.22 | 0.19 | 0.14 | 10 | 24.7 |
| value | −0.34 | −0.37 | −0.41 | 9 | −35.5 |
| 1/N blend | 0.12 | −0.38 | −1.21 | 145 | 2.7 |
| **mean-reversion (4 d)** | **0.13** | **−0.32** | **−1.05** | **159** | **2.9** |

### What works, and why

- **Momentum** is the best net signal. Its IC is the highest at 1 and 21 days (0.016 and 0.015).
  It trades 13 times a year, and its break-even of 33 bp leaves a comfortable margin above 10 bp.
  It is a slow signal: it pays little in costs.
- **Quality** works for the same reason. Its IC grows with the horizon (0.005 at 1 day, 0.013 at
  63 days), which is consistent with a fundamental signal that only changes at each filing.

### What does not work, and why

- **Value** loses money **even before costs** (−3.3%/yr gross). Its IC is negative at every
  horizon and becomes more negative as the horizon lengthens (−0.015 at 63 days). This is not an
  execution problem. On US large caps from 2011 to 2024, expensive stocks (growth, technology)
  beat cheap ones. Value is also negatively correlated with momentum (−0.23): it is short the
  winners.
- **Mean-reversion** predicts (IC of 0.007 to 0.010, gross Sharpe around 0.6 at any lag ≥ 0), but
  its gross edge is only worth 1.2 bp per unit traded. That is less than the most optimistic cost.
  The lag test confirms there is no look-ahead: `lag=-1` gives a Sharpe of −21, while `lag` from
  0 to 2 gives between 0.62 and 0.64.
- **The 1/N blend** does not benefit from diversification. Mean-reversion is a quarter of the
  portfolio but most of its trading (turnover 145 versus 9 to 13 for the slow signals). The blend
  does worse than momentum alone at every cost level.

### The improvement

Rebalancing mean-reversion every 4 days instead of every day cuts turnover from 305 to 159 times
a year. The break-even rises from 1.2 to 2.9 bp, and the net Sharpe at 2 bp goes from −0.24 to
+0.13. But it stays negative at 5 bp (−0.32) and at 10 bp (−1.05). This gain should be taken with
caution:

- the frequency was chosen on the whole sample, with no out-of-sample test;
- the results are not monotonic (3 days: −0.15; 5 days: −0.17 at 2 bp), which suggests that part
  of the gain at 4 days is noise.

### Limitations

- **Significance.** Over 14 years, the standard error of an annual Sharpe is about
  1/√14 ≈ 0.27. Momentum's net Sharpe of 0.35 is therefore about 1.3 standard errors from zero:
  it is not statistically significant.
- **Costs.** The model is linear: no market impact, no borrow cost on short positions, no spread
  varying by stock.
- **Delistings.** A position in a stock that leaves the tape is marked at zero on the last day
  (see §7): the acquisition or bankruptcy price is not in the data.
- **Sample.** A single universe (US top 500), a single period, and every parameter was looked at
  on the full sample.

### Difficulties encountered

- **Dating conventions.** Separating the signal date, the execution date and the holding date.
  Only then does the P&L of day t correspond to the return of t+1, and σ stop at t−1.
- **Point-in-time fundamentals.** Attaching them to the filing date rather than the quarter end,
  and only summing adjacent quarters in `trailing`.
- **Returns.** Moving from price return to total return (`trfd`), so that spin-offs and dividends
  are not read as price moves.
- **Computation time.** The portfolio construction loop makes every parameter grid slow on
  2 million rows.

### Next steps, with six months

1. **Out-of-sample validation.** Choose the rebalancing frequency walk-forward, then test on 2025,
   which has never been looked at.
2. **Cheaper mean-reversion.** A no-trade band (only trade when the score has changed enough), a
   volatility-scaled score, a sector-neutral version.
3. **Blend.** Weight the signals by break-even or risk rather than 1/N, or exclude a signal while
   its break-even is below the cost.
4. **Value.** A sector-relative version and an adjustment for intangible assets, to separate
   "cheap" from "declining".
5. **Cost model.** Add market impact (square root of the traded volume) and borrow costs, then
   re-read every break-even.
6. **Monitoring.** Wire the six indicators of §11 to daily data and calibrate their thresholds
   out of sample.